In [1]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith
from pathlib import Path

import pandas as pd

from datasmith.benchmark.collection import BenchmarkCollection

/mnt/sdd1/atharvas/formulacode/datasmith


22:26:31 WARNING  simple_useragent.core: Falling back to historic user agent.


In [2]:
collections = [BenchmarkCollection.load(p) for p in Path("scratch/artifacts/processed").rglob("*breakpoints.fc.pkl")]
list(Path("scratch/artifacts/processed").rglob("*breakpoints.fc.pkl"))

[PosixPath('scratch/artifacts/processed/downloads/numpy/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/distributed/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/pymc3/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/joblib/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/sklearn/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/pandas/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/pandas2/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/scikit-image/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/dask/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/astropy/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/xarray/breakpoints.fc.pkl'),
 PosixPath('scratch/artifacts/processed/downloads/scipy/breakpoints.fc.pkl')]

In [ ]:
bps = []
for c in collections:
    df = c.enriched_breakpoints
    df["repo_name"] = f"{c.task.owner}/{c.task.repo}"
    commits = c.commits[["files_changed", "sha", "file_change_summary", "message", "patch", "repo_name"]].rename(
        columns={"sha": "gt_hash"}
    )
    frame = (
        df.groupby(["repo_name", "hash", "gt_hash"])["delta_pct"]
        .mean()
        .reset_index()
        .rename(columns={"delta_pct": "delta_pct_mean"})
    )

    merged_frame = frame.merge(commits, on=["repo_name", "gt_hash"], how="left")
    assert all(merged_frame.notnull().all())  # noqa: S101
    bps.append(merged_frame)

all_enriched = pd.concat(bps, ignore_index=True)
all_enriched

,repo_name,hash,gt_hash,delta_pct_mean,files_changed,file_change_summary,message,patch
0,numpy/numpy,00a45b4dca164105b50ba29e1735e96b573b639c,b0222e05b9b98395fcdab4e6ac7f5fed518ec387,-1.011089,numpy/core/setup.py,| File | Lines Added | Lines Removed | Total C...,MAINT: Remove deplicated symbols from link ste...,From b0222e05b9b98395fcdab4e6ac7f5fed518ec387 ...
1,numpy/numpy,024cfe13767ad8a5e6a2693ed9ff56df1d122437,26b2a5c0b05ddd9add6b412f967a968c65f13bb7,-1.406231,doc/source/_templates/autosummary/attribute.rs...,| File | Lines Added | Lines Removed | Total C...,Merge pull request #11347 from mattip/less-sph...,From 2804c03bdc135b70cbcc24755d450123274b4850 ...
2,numpy/numpy,1405a30b1f1100f88c38731a9170f889002d316a,4a54bc458b93adbea75cb3ba05978ab327ff1552,-1.759909,numpy/core/overrides.py\nnumpy/random/bit_gene...,| File | Lines Added | Lines Removed | Total C...,Merge pull request #16798 from Carreau/rst-min...,From f5b1a67fcc4ad27bce504948c79e606aa06b64c0 ...
3,numpy/numpy,1713503b4117550152ca47feee651e67275f3557,570b398a1f49571b13838800007a214dccb34d07,-0.618666,setup.py,| File | Lines Added | Lines Removed | Total C...,Merge pull request #17990 from charris/fix-f-s...,From e41dfe89958eb2e212fd60b174aada41786a7723 ...
4,numpy/numpy,2ac14a153ed79377130a38e1425574e97602455e,36fd10c9c261cef871d2c86b91d55c0502fa64d2,-1.184804,test_requirements.txt,| File | Lines Added | Lines Removed | Total C...,MAINT: Bump hypothesis from 5.8.0 to 5.8.3\n\n...,From 36fd10c9c261cef871d2c86b91d55c0502fa64d2 ...
...,...,...,...,...,...,...,...,...
876,scipy/scipy,fbfd96c9bd210d2c83d470ca866086382b7fe913,3dddff391aec8c076ba93a504f8050fb0fd7fba3,-0.348127,scipy/spatial/ckdtree.pyx\nscipy/spatial/ckdtr...,| File | Lines Added | Lines Removed | Total C...,Merge pull request #12334 from peterbell10/ckd...,From e6bf56684fbd9c83a6a67ab927275037ac56c212 ...
877,scipy/scipy,fc58ce9e472a61da0c31ca84d17ab4f60bce3ae7,47a96c841ed1e9813afa570d1da696c4942909a4,-53.229922,scipy/_build_utils/_fortran.py,| File | Lines Added | Lines Removed | Total C...,BLD: separate build hook for ilp64 fortran fla...,From 47a96c841ed1e9813afa570d1da696c4942909a4 ...
878,scipy/scipy,fd3e8c0b871f6033bfd779e69ded86944f7f039c,ff72ef8d8ef64e9a125d3fca0dec1c0cb5b6c38d,-0.307288,pavement.py\nsetup.py,| File | Lines Added | Lines Removed | Total C...,REL: set version to 1.5.0rc1\n,From ff72ef8d8ef64e9a125d3fca0dec1c0cb5b6c38d ...
879,scipy/scipy,fdbe641f9cc38c1b2b97d980f3486a813842de4c,771b3ad774fac08788c1203cb475a384820088f9,-1.479340,scipy/stats/__init__.py\nscipy/stats/_hypotest...,| File | Lines Added | Lines Removed | Total C...,Merge pull request #13263 from chrisb83/cvm_2s...,From cb497aff16a119cb4b584a99760ccc2602a8aad9 ...


In [4]:
useful_enriched = all_enriched[(-1 * all_enriched["delta_pct_mean"]) > 10]
print(f"Found {len(useful_enriched)} tasks with >5% mean improvement in at least one commit")
useful_enriched["repo_name"].value_counts().reset_index()

Found 58 tasks with >5% mean improvement in at least one commit


,repo_name,count
0,pandas-dev/pandas,23
1,scipy/scipy,21
2,astropy/astropy,6
3,numpy/numpy,6
4,joblib/joblib,1
5,scikit-learn/scikit-learn,1


In [5]:
print(useful_enriched.columns)
useful_enriched.head()

Index(['repo_name', 'hash', 'gt_hash', 'delta_pct_mean', 'files_changed',
       'file_change_summary', 'message', 'patch'],
      dtype='object')


,repo_name,hash,gt_hash,delta_pct_mean,files_changed,file_change_summary,message,patch
7,numpy/numpy,3b3dbdda2de4d6193dd4ab299067f20fb47515f2,86b9cedb9ac3330ed8c52cebb48f38bfd87298e0,-13.732516,numpy/core/_add_newdocs.py,| File | Lines Added | Lines Removed | Total C...,Merge pull request #19093 from paxcodes/add_re...,From d7a1004058e7baac86f732699cbed975b0a16613 ...
9,numpy/numpy,3c91a3e1704c8aa7f1258aa30892040df9d952f4,fe3d717080e812383900507168a6bd7093c4e434,-70.867376,test_requirements.txt,| File | Lines Added | Lines Removed | Total C...,Merge pull request #19082 from numpy/dependabo...,From 97bce62558cdfd75395bc271f3263d9e71702612 ...
14,numpy/numpy,5772f450ea818dc41c7e7b97582852193caa4e80,58a9900e10a64d44de4c288ef2867ecd0d09a206,-20.362328,numpy/core/src/multiarray/arraytypes.c.src\nnu...,| File | Lines Added | Lines Removed | Total C...,Merge pull request #8976 from eric-wieser/void...,From 0107956102d914a68f157d80614f207f78dffb96 ...
17,numpy/numpy,612cd65633eefea009b9f1aae6605b73523259d2,5e5fc968892999b13bbbcbcf0bbe2605ef745139,-90.613945,numpy/core/src/multiarray/common.h,| File | Lines Added | Lines Removed | Total C...,BUG: add case for longdouble alignment size\n,From 5e5fc968892999b13bbbcbcf0bbe2605ef745139 ...
21,numpy/numpy,84f6c85cc2c8b1ec5e4712a163653d47e7c0888c,b428d4b2c310a9932bdec730b7b5e8d4f327a4d2,-12.997208,numpy/f2py/src/fortranobject.c,| File | Lines Added | Lines Removed | Total C...,Merge pull request #13274 from charris/backpor...,From 95bc2f0609fb858ae85af43ab018f7fd367d075e ...


In [ ]:
useful_enriched.to_csv("scratch/artifacts/processed/useful_commits.csv", index=False)

In [ ]:
from datasmith.docker.context import ContextRegistry

cr1 = ContextRegistry.load_from_file(Path("scratch/merged_context_registry_2025-09-04T08:32:08.486247.json"))
cr2 = ContextRegistry.load_from_file(Path("scratch/merged_context_registry_2025-09-09T14:32:37.382974.json"))